# MethaneScope AI · Lesson 2.6
## Operational monitoring & safe run-once ingestion
**Objective:** observe the real NASA EMIT-to-PostGIS ingestion without changing existing database tables. The results are engineering quality checks, **not independent scientific validation**.

### 1. What we already proved
- 1 real NASA EMIT candidate granule and 1 GeoJSON feature stored in PostGIS.
- Repeating an identical granule returns `SKIPPED`.
- 5/5 Lesson 2.5 tests passed.

### What we add
Read-only health checks, an on-demand wrapper for validation+ingestion, local run history, structured error categories, and a JSON monitoring report.

In [ ]:
from pathlib import Path
import sys, subprocess, json

ROOT = Path.cwd()
assert (ROOT / "methanescope" / "monitor.py").is_file(), "Open JupyterLab from MethaneScope project root"
print("Project:", ROOT)
print("Python:", sys.executable)

### 2. Automated unit tests
Run the existing and new tests. These do not require live NASA downloads or modify your database.

In [ ]:
proc = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_science.py", "tests/test_monitor.py"], cwd=ROOT, text=True, capture_output=True)
print(proc.stdout, proc.stderr)
print("Exit code:", proc.returncode)

### 3. Read-only health check
This confirms NASA input matching, acceptance manifest, and PostgreSQL connectivity. Docker Desktop/PostGIS must be running. No database writes happen in this cell.

In [ ]:
proc = subprocess.run([sys.executable, "-m", "methanescope.monitor", "health"], cwd=ROOT, text=True, capture_output=True)
print(proc.stdout, proc.stderr)
print("Exit code:", proc.returncode)

### 4. Safe monitored ingestion (run once)
Run only when health is OK. The existing NASA observation should be `SKIPPED`, because Lesson 2.4 already inserted it. This step generates local audit files.

In [ ]:
if proc.returncode == 0:
    run = subprocess.run([sys.executable, "-m", "methanescope.monitor", "run-once"], cwd=ROOT, text=True, capture_output=True)
    print(run.stdout[-5000:], run.stderr)
    print("Exit code:", run.returncode)
else:
    print("Preflight was not OK; investigate health output before ingestion.")

### 5. Generate a monitoring report
The report contains the last run, recent execution counts, and the latest available ingestion audit.

In [ ]:
report = subprocess.run([sys.executable, "-m", "methanescope.monitor", "report"], cwd=ROOT, text=True, capture_output=True)
print(report.stdout, report.stderr)
print("Exit code:", report.returncode)

In [ ]:
path = ROOT / "outputs" / "monitoring" / "last_run.json"
if path.exists():
    doc = json.loads(path.read_text(encoding="utf-8"))
    print("Run result:", doc.get("result"))
    print("Elapsed seconds:", doc.get("duration_seconds"))
    print("Failure class:", doc.get("failure_class"))
else:
    print("No monitored ingestion run yet.")

### Checkpoint
1. Why is `SKIPPED` different from failure?
2. What does a CRITICAL preflight result mean?
3. Why should we keep a run history?
4. What is the difference between measurement validity and pipeline reliability?
5. Why do we not automatically schedule jobs until the pipeline is stable?

**Submit:** screenshots or text showing test results, `health` status, and `run-once` status.